# Retrieval Evaluation Tutorial

This notebook dives into basic concepts to evaluate document retrieval. In particular, it compares classic measures (precision, recall) vs. LLM as a judge.

In [1]:
from rag_llm_applications.chat import get_chat_generator
from rag_llm_applications.embed import get_embedder
from rag_llm_applications.model import Document
from rag_llm_applications.qdrant import instantiate_qclient, search_documents
from rag_llm_applications.util import PROJECT_ROOT, load_config

from pathlib import Path

import numpy as np

## 1. "Ground Truth" Evaluation

Here, we try to use classic measures and pre-defined "golden" (true) documents

In [2]:
class RetrievalEvaluator:

    def compute_precision(self, retrieved_documents, golden_documents):
      # compute the percentage of retrieved documents found in the golden docs
      return len(set(retrieved_documents).intersection(golden_documents)) / len(retrieved_documents)

    def compute_recall(self, retrieved_documents, golden_documents):
      # compute the percentage of golden documents found in the retrieved docs
      return len(set(retrieved_documents).intersection(golden_documents)) / len(golden_documents)

    def compute_mean_average_precision(self, retrieved_documents, golden_documents):
      # check which among the retrieved docs is found in the gold, keeping the order
      correct_retrieved_documents = [1 if x in golden_documents else 0 for x in retrieved_documents]
      # compute map
      map = np.mean([sum(correct_retrieved_documents[: i + 1]) / (i + 1) for i, v in enumerate(correct_retrieved_documents) if v == 1])
      return map

    def run_evals(self, retrieved_documents, golden_documents):
      precision = round(self.compute_precision(retrieved_documents, golden_documents),2)
      recall = round(self.compute_recall(retrieved_documents, golden_documents),2)
      map = round(self.compute_mean_average_precision(retrieved_documents, golden_documents),2)
      results = {'precision': [precision],
                 'recall': [recall],
                 'map': [map]}
      for k,v in results.items():
          print(f"{k}: {v[0]}")

In [4]:
config = load_config(PROJECT_ROOT / "config" / "config.yml")
qdrant_config = config["qdrant"]
qdrant = instantiate_qclient(PROJECT_ROOT / "data" / "qdrant")

In [5]:
embedder = config["embedder"]
embedder = get_embedder(embedder)

In [6]:
user_input = "What is a trust council?"

In [7]:
embedded_question = embedder.embed_query(user_input)
# embedded_question

In [8]:
top_10_documents = search_documents(
    qdrant, qdrant_config["document_collection"], embedded_question
)
top_10_documents

[ScoredPoint(id='c671ad6c-d168-4d50-9316-58592e8b0b08', version=0, score=0.8398055063409562, payload={'content': 'Geschäftsordnung Trust Council [at] \n \nDer Trust Council (TC) der Firma Alexander Thamm GmbH hat in seiner Sitzung am 16.11.2023 mit der \nStimmenmehrheit seiner Mitglieder nachfolgende Geschäftsordnung beschlossen: \n§ 1 Zweck \nDie Geschäftsordnung wurde beschlossen um den Arbeitsmodus des TC festzulegen.  \n§ 2 Aufgaben des TC-Vorsitzenden \nSoweit nicht anders vereinbart, führt der/die Vorsitzende des TC die laufenden Geschäfte und \nübernimmt die Umsetzung von Beschlüssen. \nDazu gehören beispielsweise: \n• \ndie Vorbereitung von Sitzungen \n• \ndie Erteilung und der Entzug des Rederechts in Trust Council Sitzungen \n• \nder anfallende Schriftwechsel mit Geschäftsleitung, juristischen Beratern sowie weiteren \nParteien \n• \ndie Vertretung des Trust Council nach außen \nDie obigen Aufgabengebiete können auch an andere Mitglieder des TC delegiert werden. \n§ 3 Vertret

In [14]:
Path(top_10_documents[0].payload.get("source")).name # get the source of the first document

'Geschäftsordnung  Trust Council.pdf'

In [16]:
golden_docs = ["Geschäftsordnung  Trust Council.pdf","Gesellschafterbeschluss Trust Council.pdf","Rules of Procedure Trust Council.pdf"]

In [15]:
retrieved_docs = []
retrieved_docs = [Path(top_10_documents[i].payload.get("source")).name 
                  for i in range(len(top_10_documents))]
retrieved_docs = list(set(retrieved_docs))
print(retrieved_docs)

['Rules of Procedure Trust Council.pdf', 'Geschäftsordnung  Trust Council.pdf', 'German Works Council_ Everything You Need to Know.pdf']


In [12]:
# we can now instantiate the evaluator
evaluate_retrieval = RetrievalEvaluator()

# and run the evaluation
evaluate_retrieval.run_evals(retrieved_docs,golden_docs)

precision: 0.67
recall: 0.67
map: 0.83


### Task 1.1 Try to define different questions and true documents. How well is our retrieval generally doing?

## 2. LLM as a judge

Another common method is to let the relevance being judged by an LLM. Why not let an LLM solve our uncertainty about an LLM application, what could possibly go wrong?

In [17]:
chat_generator = config["chat_generator"]
chat_generator = get_chat_generator(chat_generator)

In [18]:
input_prompt = "Hi hi how are you?"

In [19]:
response = chat_generator.invoke(input_prompt)
print(response.content)

Hello! I'm just a computer program, so I don't have feelings, but I'm here to help you. How can I assist you today?


In [20]:
retrieved_content = [top_10_documents[i].payload.get("content") for i in range(len(top_10_documents))]

In [17]:
retrieved_content

['Geschäftsordnung Trust Council [at] \n \nDer Trust Council (TC) der Firma Alexander Thamm GmbH hat in seiner Sitzung am 16.11.2023 mit der \nStimmenmehrheit seiner Mitglieder nachfolgende Geschäftsordnung beschlossen: \n§ 1 Zweck \nDie Geschäftsordnung wurde beschlossen um den Arbeitsmodus des TC festzulegen.  \n§ 2 Aufgaben des TC-Vorsitzenden \nSoweit nicht anders vereinbart, führt der/die Vorsitzende des TC die laufenden Geschäfte und \nübernimmt die Umsetzung von Beschlüssen. \nDazu gehören beispielsweise: \n• \ndie Vorbereitung von Sitzungen \n• \ndie Erteilung und der Entzug des Rederechts in Trust Council Sitzungen \n• \nder anfallende Schriftwechsel mit Geschäftsleitung, juristischen Beratern sowie weiteren \nParteien \n• \ndie Vertretung des Trust Council nach außen \nDie obigen Aufgabengebiete können auch an andere Mitglieder des TC delegiert werden. \n§ 3 Vertretung des/der TC-Vorsitzenden \n1. Im Falle der Verhinderung des/der TC-Vorsitzenden nimmt der/die stellvertretend

In [21]:
input_prompt = f"""
The user question was: {user_input}

The retrieved content is:
{retrieved_content}

Please judge which of the retrieved content chunks are relevant to the user question and which are not.
Please answer with a list of the relevant chunks, and a list of the non-relevant chunks.

Go through every item in the list of retrieved chunks and check if it is relevant to the user question.
"""

In [22]:
response = chat_generator.invoke(input_prompt)

In [23]:
from IPython.display import Markdown, display
display(Markdown(response.content))

Relevant chunks:
1. Rules of Procedure Trust Council
2. What Is the Purpose of a German Works Council?
3. Advantages of German Works Councils

Non-relevant chunks:
1. Geschäftsordnung Trust Council
2. While global expansion has many advantages...
3. 5. the TC chairperson is responsible for ensuring...
4. dieser Anträge eine gesonderte Abstimmung statt...
5. Is a Works Council Mandatory in Germany?
6. Who Are the Members of a German Works Council?

### Tasks 2.1 How well is this method working? Are the answers generally correct? What does it say about the retrieval quality?

### Task 2.2 Can you implement this in our RAG application?

## 3. BONUS: LLM Agent as a Judge

Bonus: We can implement the above LLM as a judge idea in an LLM *agent*

In [24]:
from langchain.chat_models import ChatOpenAI

from langchain.agents import AgentExecutor, create_openai_tools_agent # tools for creating and running agents based on openai tools

from langchain_core.messages import (
    BaseMessage,
    HumanMessage,
    AIMessage
) # message classes, see https://python.langchain.com/v0.1/docs/modules/model_io/chat/message_types/

from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder

import textwrap # for wrapping text

import functools

In [25]:
def create_agent(llm: ChatOpenAI, tools: list, system_prompt: str):
    prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                system_prompt,
            ),
            MessagesPlaceholder(variable_name="messages"), # sequence of messages that contain the previous messages
            MessagesPlaceholder(variable_name="agent_scratchpad"), # contains the previous agent tool calls and outputs
        ]
    )
    agent = create_openai_tools_agent(llm, tools, prompt) # create an agent that uses the openai tools based on the prompt
    executor = AgentExecutor(agent=agent, tools=tools) # create an executor that can run the agent and tools
    return executor

In [26]:
llm = ChatOpenAI(model="gpt-3.5-turbo")

/var/folders/vd/46lkd9v91yl4hf2mpj_mgjlh0000gq/T/ipykernel_50227/2690798537.py:1: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import ChatOpenAI``.
  llm = ChatOpenAI(model="gpt-3.5-turbo")


In [27]:
retrieval_review_agent = create_agent(
    llm,
    [],
    """
    You are an assitant who evaluates the quality of retrieved text with respect to a user question.
    When you are asked to evaluate the qulity of retrieved content, output which content is relevant and which is not, plus an explanation of why you think so.
    """,
)

In [28]:
input_prompt = f"""
The user question was: {user_input}

The retrieved content is:
{retrieved_content}

Please judge which of the retrieved content chunks are relevant to the user question and which are not.
Please answer with a list of the relevant chunks, and a list of the non-relevant chunks.

Go through every item in the list of retrieved chunks and check if it is relevant to the user question.
"""

In [30]:
output = retrieval_review_agent.invoke({"messages": [HumanMessage(content=input_prompt)]})

BadRequestError: Error code: 400 - {'error': {'message': "This model's maximum context length is 16385 tokens. However, your messages resulted in 16587 tokens (16574 in the messages, 13 in the functions). Please reduce the length of the messages or functions.", 'type': 'invalid_request_error', 'param': 'messages', 'code': 'context_length_exceeded'}}

In [ ]:
print(f"Question: {output['messages'][0].content}")

print(f"Answer: {output['output']}")

### Bonus Task 3.1 Can you implement this agent in our RAG application?

### Bonus Task 3.2 What other agents could be useful in our RAG application? Can you implement some of them?